# IoT Intrusion Detection: Dataset Walkthrough

This notebook matches the current project pipeline:

1. Load the prepared sample from `data/processed/sample.parquet`
2. Compare the dataset labels to the schema in `src/schema.py`
3. Clean the dataset with the shared helpers in `src/data_pipeline.py`
4. Apply the notebook log transform for skewed features
5. Build train/test splits and oversample only the training split


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_pipeline import (
    build_training_splits,
    clean_dataset,
    describe_split,
    load_dataset,
    log_transform_features,
    summarize_dataset_labels,
)
from src.schema import ATTACK_CLASSES, LABEL_COLUMN

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Load The Prepared Sample

If this cell raises `FileNotFoundError`, run `python data/fetch/fetch_sample.py` or `python data/fetch/prepare_sample.py` first.

In [ ]:
df_raw = load_dataset()
print("Dataset shape:", df_raw.shape)
df_raw.head()

## Label Coverage vs Schema

The sample can contain labels outside the current 10-class schema. This summary shows what matches and what does not.

In [ ]:
label_summary = pd.Series(summarize_dataset_labels(df_raw))
label_summary

In [ ]:
raw_label_counts = df_raw[LABEL_COLUMN].value_counts().rename_axis(LABEL_COLUMN).rename("count")
raw_label_counts.to_frame().head(15)

## Restrict To The Current Schema

For training experiments that align with the current API/schema, keep only labels listed in `ATTACK_CLASSES`.

In [ ]:
df_schema = clean_dataset(df_raw, allowed_labels=ATTACK_CLASSES)

summary_table = pd.DataFrame(
    {
        "rows": [len(df_raw), len(df_schema)],
        "label_count": [df_raw[LABEL_COLUMN].nunique(), df_schema[LABEL_COLUMN].nunique()],
    },
    index=["raw_sample", "schema_filtered"],
)
summary_table

In [ ]:
schema_label_counts = (
    df_schema[LABEL_COLUMN]
    .value_counts()
    .reindex(ATTACK_CLASSES, fill_value=0)
    .rename("count")
)
schema_label_counts.to_frame()

## Feature Cleanup And Log Transform

`clean_dataset()` handles missing labels, duplicates, and non-numeric feature values. `log_transform_features()` applies `log1p` to the skewed notebook features.

In [ ]:
df_transformed = log_transform_features(df_schema)

feature_summary = pd.DataFrame(
    {
        "schema_flow_duration_mean": [df_schema["flow_duration"].mean()],
        "transformed_flow_duration_mean": [df_transformed["flow_duration"].mean()],
        "schema_rate_mean": [df_schema["rate"].mean()],
        "transformed_rate_mean": [df_transformed["rate"].mean()],
    }
)
feature_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_schema["flow_duration"], bins=50, color="#4C78A8", alpha=0.85)
axes[0].set_title("flow_duration before log1p")
axes[0].set_xlabel("flow_duration")
axes[0].set_ylabel("rows")

axes[1].hist(df_transformed["flow_duration"], bins=50, color="#F58518", alpha=0.85)
axes[1].set_title("flow_duration after log1p")
axes[1].set_xlabel("log1p(flow_duration)")

plt.tight_layout()
plt.show()

## Build Training Splits

`build_training_splits()` performs the schema-aligned clean/split flow and oversamples only the training set to reduce leakage risk.

In [ ]:
splits = build_training_splits(allowed_labels=ATTACK_CLASSES, test_size=0.2, random_state=42)

print("X_train:", splits.X_train.shape)
print("X_test:", splits.X_test.shape)
describe_split(splits.y_train, splits.y_test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

splits.y_train.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#54A24B")
axes[0].set_title("Training labels after oversampling")
axes[0].set_xlabel(LABEL_COLUMN)
axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=45)

splits.y_test.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#E45756")
axes[1].set_title("Test labels without oversampling")
axes[1].set_xlabel(LABEL_COLUMN)
axes[1].set_ylabel("rows")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()